# Quadmesh 3B — Training Commands

Plain commands, one per cell, `!` prefix so they run as shell commands in the notebook kernel.
Each cell's output is saved with the notebook when you save it (`Ctrl+S`) — that's your recorded
command + log, no wrapper code needed.

`--detach` cells return almost immediately (the job keeps running on Modal); they do **not** block
until training finishes. Re-run the "check on it" cells in section 5 any time to pull fresh logs.

## 1. Setup

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -r requirements-local-agent.txt

In [ ]:
!python tests/run_offline_tests.py

## 2. Accounts

In [ ]:
!huggingface-cli login

In [ ]:
!modal setup

In [ ]:
!modal secret create huggingface-secret HF_TOKEN=<YOUR-TOKEN>

## 3. Tokenizer + review queue (once)

In [ ]:
!modal run --detach modal_app.py::build_tokenizer

Same pattern as training: this returns immediately with an app id, it does not
block until the tokenizer finishes. Check on it with the cells below - re-run them any time for a fresh
snapshot. The dashboard (https://modal.com/apps) always shows live progress too, regardless of what
shows here.

In [ ]:
# List running/recent apps to find the tokenizer job's app id
!modal app list

In [ ]:
# Replace with the actual app id from the list above
APP_ID = "ap-XXXXXXXXXXXX"
!modal app logs {APP_ID}

Text2CAD is a gated dataset - if you see "Finished Text2CAD stream (0 docs)" in
the log, you have not accepted its license yet. Go to
https://huggingface.co/datasets/SadilKhan/Text2CAD with the same HF account as your token, click
"Agree and access repository", then re-run the `build_tokenizer` cell above.

Once the log shows `[tokenizer] All ... docs buffered! Calculating BPE merges...`
followed by it finishing, the tokenizer is done and cached on the volume - you only need to do this
once, not before every training run.

In [ ]:
!modal run modal_app.py::shared_review_queue

In [ ]:
!modal volume get quadmesh-ckpts /data/review_queue.jsonl review_queue.jsonl

Run this one in a separate terminal, not the notebook - it's interactive:

`python -m quadmesh.pipeline.review_queue review_queue.jsonl`

In [ ]:
!modal volume put quadmesh-ckpts review_queue.jsonl /data/review_queue.jsonl --force

## 4. Train — chinchilla budget first (cheap validation pass)

In [ ]:
!modal run --detach modal_app.py::shared_pretrain --tier 3B --budget chinchilla --mb 16 --accum 16 --grad_ckpt 0 --workers 12

Wait for the cell above's run to reach `done=true` (check in section 5) before running SFT below - SFT loads the pretrain checkpoint as its starting point.

In [ ]:
!modal run --detach modal_app.py::shared_sft --tier 3B --tokens 2000000000 --mb 16 --accum 16 --grad_ckpt 0 --workers 12

In [ ]:
!modal run modal_app.py::shared_eval --tier 3B --n 100

## 5. Check on it while it runs (re-run any time for a fresh log snapshot)

In [ ]:
!modal app list

In [ ]:
# replace <APP_ID> with the id from the cell above
!modal app logs <APP_ID>

In [ ]:
!modal volume get quadmesh-ckpts /Quadmesh-3B/live/metrics_pretrain.jsonl metrics_pretrain.jsonl --force

In [ ]:
!type metrics_pretrain.jsonl

In [ ]:
!modal volume get quadmesh-ckpts /Quadmesh-3B/live/metrics_sft.jsonl metrics_sft.jsonl --force

In [ ]:
!type metrics_sft.jsonl

## 6. Download + build the final report once training shows `done: true`

In [ ]:
!modal volume get quadmesh-ckpts /Quadmesh-3B ./Quadmesh-3B

In [ ]:
!python -m quadmesh.pipeline.training_report --tier 3B --ckpt-dir ./Quadmesh-3B --out training_report.md

In [ ]:
!type training_report.md

## 7. Only if chinchilla's results justify it — full 4x run

Same commands, `--budget 4x` instead of `chinchilla`. Overwrites the chinchilla checkpoint (same fixed filename on the volume) - a fresh longer run, not additive.

In [ ]:
!modal run --detach modal_app.py::shared_pretrain --tier 3B --budget 4x --mb 16 --accum 16 --grad_ckpt 0 --workers 12

In [ ]:
!modal run --detach modal_app.py::shared_sft --tier 3B --tokens 2000000000 --mb 16 --accum 16 --grad_ckpt 0 --workers 12

In [ ]:
!modal run modal_app.py::shared_eval --tier 3B --n 100